In [2]:
# 03. 장바구니 이탈 분석
# view → cart까지 진행한 세션×상품을 대상으로 구매 전환 여부와 행동 차이를 분석

In [7]:
# DuckDB 불러오기 및 10월·11월 Parquet 데이터 경로 설정
import duckdb

parquet_path = r"..\data\processed\2019-*.parquet"

In [9]:
# view → cart까지 진행한 세션×상품을 구매 완료/장바구니 이탈로 구분
cart_analysis = duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,

            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY user_session, product_id
    )

    SELECT
        user_session,
        product_id,
        first_view_time,
        first_cart_time,
        first_purchase_time,

        CASE
            WHEN first_purchase_time IS NOT NULL
             AND first_cart_time <= first_purchase_time
            THEN 1
            ELSE 0
        END AS converted

    FROM event_times

    WHERE first_view_time IS NOT NULL
      AND first_cart_time IS NOT NULL
      AND first_view_time <= first_cart_time
""")

cart_analysis.limit(10).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────┬────────────┬─────────────────────┬─────────────────────┬─────────────────────┬───────────┐
│             user_session             │ product_id │   first_view_time   │   first_cart_time   │ first_purchase_time │ converted │
│               varchar                │   int64    │      timestamp      │      timestamp      │      timestamp      │   int32   │
├──────────────────────────────────────┼────────────┼─────────────────────┼─────────────────────┼─────────────────────┼───────────┤
│ c450dca9-5fbd-44df-b9c0-9c1cc823323f │    1004766 │ 2019-11-08 05:56:12 │ 2019-11-08 05:57:29 │ 2019-11-08 05:59:56 │         1 │
│ fb160a72-731e-407d-a189-5414096f3d69 │   10900306 │ 2019-11-08 06:03:24 │ 2019-11-08 06:04:11 │ 2019-11-08 06:05:00 │         1 │
│ be3094f2-5538-4cc6-9cde-aa0283663324 │    1005160 │ 2019-11-08 06:07:24 │ 2019-11-08 06:08:13 │ NULL                │         0 │
│ 3ce3e85d-3d1c-4056-9459-e01454896e58 │    5500161 │ 2019-11-08 06:08:36 │ 

In [10]:
# 장바구니 진입 후 구매 완료와 이탈 건수를 비교
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 'purchase'
                ELSE 'abandon'
            END AS cart_result
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    )
    SELECT
        cart_result,
        COUNT(*) AS session_product_count
    FROM cart_cases
    GROUP BY cart_result
    ORDER BY session_product_count DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬───────────────────────┐
│ cart_result │ session_product_count │
│   varchar   │         int64         │
├─────────────┼───────────────────────┤
│ abandon     │               1338158 │
│ purchase    │               1001613 │
└─────────────┴───────────────────────┘



In [11]:
# 확인 결과
# - 조회 → 장바구니까지 진행한 세션×상품 중
#   장바구니 이탈 1,338,158건, 구매 완료 1,001,613건
# - 장바구니 이후 구매하지 않은 사례가 충분히 많아 이탈 원인 비교 분석이 가능함

In [12]:
# 장바구니 이탈 상품과 구매 완료 상품의 평균·중앙 가격 비교
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time,
            MAX(price) AS price
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            price,
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 'purchase'
                ELSE 'abandon'
            END AS cart_result
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    )
    SELECT
        cart_result,
        COUNT(*) AS cases,
        ROUND(AVG(price), 2) AS avg_price,
        ROUND(MEDIAN(price), 2) AS median_price
    FROM cart_cases
    GROUP BY cart_result
    ORDER BY cart_result
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────┬───────────┬──────────────┐
│ cart_result │  cases  │ avg_price │ median_price │
│   varchar   │  int64  │  double   │    double    │
├─────────────┼─────────┼───────────┼──────────────┤
│ abandon     │ 1338158 │    288.86 │        165.2 │
│ purchase    │ 1001613 │    315.81 │        180.7 │
└─────────────┴─────────┴───────────┴──────────────┘



In [13]:
# 확인 결과
# - 구매 완료 그룹의 평균·중앙 가격이 이탈 그룹보다 더 높음
# - 따라서 단순히 가격이 높아서 장바구니 이탈이 증가한다고 보기는 어려움
# - 가격대별 전환율을 추가로 확인해 비선형적인 패턴이 있는지 볼 필요가 있음

In [14]:
# 가격대를 구간별로 나누어 장바구니 이후 구매 전환율 비교
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time,
            MAX(price) AS price
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            price,
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 1
                ELSE 0
            END AS converted
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    )
    SELECT
        CASE
            WHEN price < 50 THEN '1. <50'
            WHEN price < 100 THEN '2. 50-99'
            WHEN price < 200 THEN '3. 100-199'
            WHEN price < 500 THEN '4. 200-499'
            WHEN price < 1000 THEN '5. 500-999'
            ELSE '6. 1000+'
        END AS price_range,
        COUNT(*) AS cart_cases,
        SUM(converted) AS purchases,
        ROUND(SUM(converted) * 100.0 / COUNT(*), 2) AS conversion_rate
    FROM cart_cases
    GROUP BY price_range
    ORDER BY price_range
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬───────────┬─────────────────┐
│ price_range │ cart_cases │ purchases │ conversion_rate │
│   varchar   │   int64    │  int128   │     double      │
├─────────────┼────────────┼───────────┼─────────────────┤
│ 1. <50      │     409756 │    138658 │           33.84 │
│ 2. 50-99    │     288595 │    109485 │           37.94 │
│ 3. 100-199  │     608009 │    287156 │           47.23 │
│ 4. 200-499  │     619931 │    279490 │           45.08 │
│ 5. 500-999  │     285212 │    128564 │           45.08 │
│ 6. 1000+    │     128268 │     58260 │           45.42 │
└─────────────┴────────────┴───────────┴─────────────────┘



In [21]:
# 확인 결과
# - 50 미만 저가 상품의 장바구니 이후 구매전환율이 가장 낮음
# - 100~199 가격대에서 전환율이 가장 높고, 200 이상은 약 45% 수준으로 비교적 안정적
# - 따라서 '가격이 높을수록 이탈한다'는 단순한 관계는 관찰되지 않음
# - 저가 상품군의 낮은 전환 원인은 상품군 구성이나 사용자 행동 특성 등 추가 분석 필요

In [23]:
# 구매 완료와 장바구니 이탈 그룹의 장바구니 이전 조회 횟수 비교
duckdb.sql(f"""
    WITH cart_cases AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    valid_cases AS (
        SELECT
            c.user_session,
            c.product_id,
            c.first_cart_time,
            CASE
                WHEN c.first_purchase_time IS NOT NULL
                 AND c.first_cart_time <= c.first_purchase_time
                THEN 'purchase'
                ELSE 'abandon'
            END AS cart_result
        FROM cart_cases c
        WHERE c.first_view_time IS NOT NULL
          AND c.first_cart_time IS NOT NULL
          AND c.first_view_time <= c.first_cart_time
    ),
    view_counts AS (
        SELECT
            v.cart_result,
            v.user_session,
            v.product_id,
            COUNT(*) AS views_before_cart
        FROM valid_cases v
        JOIN read_parquet('{parquet_path}') e
          ON v.user_session = e.user_session
         AND v.product_id = e.product_id
        WHERE e.event_type = 'view'
          AND e.event_time <= v.first_cart_time
          AND CAST(e.event_time AS DATE) <> '2019-11-15'
        GROUP BY
            v.cart_result,
            v.user_session,
            v.product_id
    )
    SELECT
        cart_result,
        COUNT(*) AS cases,
        ROUND(AVG(views_before_cart), 2) AS avg_views_before_cart,
        MEDIAN(views_before_cart) AS median_views_before_cart
    FROM view_counts
    GROUP BY cart_result
    ORDER BY cart_result
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────┬───────────────────────┬──────────────────────────┐
│ cart_result │  cases  │ avg_views_before_cart │ median_views_before_cart │
│   varchar   │  int64  │        double         │          double          │
├─────────────┼─────────┼───────────────────────┼──────────────────────────┤
│ abandon     │ 1338158 │                  1.67 │                      1.0 │
│ purchase    │ 1001613 │                  1.55 │                      1.0 │
└─────────────┴─────────┴───────────────────────┴──────────────────────────┘



In [25]:
# 확인 결과
# - 장바구니 이탈 그룹의 평균 사전 조회 횟수는 1.67회, 구매 그룹은 1.55회
# - 두 그룹 모두 중앙값은 1회로 동일
# - 이탈 그룹이 약간 더 많이 조회하지만 차이가 작아 핵심적인 구분 변수라고 보기는 어려움

In [27]:
# 구매 완료와 이탈 그룹의 첫 조회 → 첫 장바구니까지 걸린 시간 비교
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 'purchase'
                ELSE 'abandon'
            END AS cart_result,

            DATE_DIFF(
                'second',
                first_view_time,
                first_cart_time
            ) AS seconds_to_cart

        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    )
    SELECT
        cart_result,
        COUNT(*) AS cases,
        ROUND(AVG(seconds_to_cart), 2) AS avg_seconds_to_cart,
        MEDIAN(seconds_to_cart) AS median_seconds_to_cart,
        QUANTILE_CONT(seconds_to_cart, 0.75) AS p75_seconds_to_cart
    FROM cart_cases
    GROUP BY cart_result
    ORDER BY cart_result
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────┬─────────────────────┬────────────────────────┬─────────────────────┐
│ cart_result │  cases  │ avg_seconds_to_cart │ median_seconds_to_cart │ p75_seconds_to_cart │
│   varchar   │  int64  │       double        │         double         │       double        │
├─────────────┼─────────┼─────────────────────┼────────────────────────┼─────────────────────┤
│ abandon     │ 1338158 │              262.45 │                   38.0 │               110.0 │
│ purchase    │ 1001613 │              152.83 │                   26.0 │                79.0 │
└─────────────┴─────────┴─────────────────────┴────────────────────────┴─────────────────────┘



In [29]:
# 확인 결과
# - 구매 완료 그룹은 첫 조회 후 장바구니까지 걸리는 시간이 더 짧음
# - 중앙값 기준 abandon 38초, purchase 26초
# - 평균도 abandon 262초, purchase 153초로 차이가 큼
# - 구매 의도가 높은 사용자가 더 빠르게 장바구니 행동으로 이동하는 패턴일 가능성
# - 단, 시간 차이가 구매의 원인이라고 단정할 수는 없음

In [31]:
# 첫 조회 → 장바구니까지 걸린 시간을 구간으로 나눠 구매전환율 비교
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            DATE_DIFF('second', first_view_time, first_cart_time) AS seconds_to_cart,
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 1
                ELSE 0
            END AS converted
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    )
    SELECT
        CASE
            WHEN seconds_to_cart <= 10 THEN '1. <=10s'
            WHEN seconds_to_cart <= 30 THEN '2. 11-30s'
            WHEN seconds_to_cart <= 60 THEN '3. 31-60s'
            WHEN seconds_to_cart <= 120 THEN '4. 61-120s'
            WHEN seconds_to_cart <= 300 THEN '5. 121-300s'
            ELSE '6. 300s+'
        END AS time_range,

        COUNT(*) AS cart_cases,
        SUM(converted) AS purchases,
        ROUND(SUM(converted) * 100.0 / COUNT(*), 2) AS conversion_rate

    FROM cart_cases
    GROUP BY time_range
    ORDER BY time_range
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬───────────┬─────────────────┐
│ time_range  │ cart_cases │ purchases │ conversion_rate │
│   varchar   │   int64    │  int128   │     double      │
├─────────────┼────────────┼───────────┼─────────────────┤
│ 1. <=10s    │     500835 │    254814 │           50.88 │
│ 2. 11-30s   │     625144 │    286219 │           45.78 │
│ 3. 31-60s   │     405072 │    160105 │           39.53 │
│ 4. 61-120s  │     309628 │    114578 │           37.01 │
│ 5. 121-300s │     258921 │     96172 │           37.14 │
│ 6. 300s+    │     240171 │     89725 │           37.36 │
└─────────────┴────────────┴───────────┴─────────────────┘



In [33]:
# 확인 결과
# - 첫 조회 후 10초 이내 장바구니 진입 시 구매전환율이 50.88%로 가장 높음
# - 장바구니까지 걸리는 시간이 길어질수록 전환율이 전반적으로 하락
# - 60초 이후에는 약 37% 수준에서 비슷하게 유지
# - 빠른 장바구니 진입은 높은 구매 의도와 연관된 행동 신호일 가능성이 있음
# - 단, 시간 자체가 구매를 유발한다고 해석할 수는 없음

In [35]:
# 구매 완료/이탈 그룹이 속한 세션의 전체 조회 상품 수와 세션 행동량 비교
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            user_session,
            product_id,
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 'purchase'
                ELSE 'abandon'
            END AS cart_result
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    ),
    session_stats AS (
        SELECT
            user_session,
            COUNT(*) AS session_events,
            COUNT(DISTINCT product_id) AS session_products,
            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS session_views,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS session_carts
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session
    )
    SELECT
        c.cart_result,
        COUNT(*) AS cases,
        ROUND(AVG(s.session_events), 2) AS avg_session_events,
        ROUND(AVG(s.session_products), 2) AS avg_session_products,
        ROUND(AVG(s.session_views), 2) AS avg_session_views,
        ROUND(AVG(s.session_carts), 2) AS avg_session_carts
    FROM cart_cases c
    JOIN session_stats s
      ON c.user_session = s.user_session
    GROUP BY c.cart_result
    ORDER BY c.cart_result
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────┬────────────────────┬──────────────────────┬───────────────────┬───────────────────┐
│ cart_result │  cases  │ avg_session_events │ avg_session_products │ avg_session_views │ avg_session_carts │
│   varchar   │  int64  │       double       │        double        │      double       │      double       │
├─────────────┼─────────┼────────────────────┼──────────────────────┼───────────────────┼───────────────────┤
│ abandon     │ 1338158 │              11.39 │                 4.68 │              9.27 │              2.01 │
│ purchase    │ 1001613 │                9.9 │                 3.33 │              6.55 │              1.99 │
└─────────────┴─────────┴────────────────────┴──────────────────────┴───────────────────┴───────────────────┘



In [37]:
# 확인 결과
# - 장바구니 이탈 그룹은 구매 그룹보다 세션 내 전체 행동량과 조회 상품 수가 더 많음
# - 평균 조회 상품 수: abandon 4.68개, purchase 3.33개
# - 평균 조회 수: abandon 9.27회, purchase 6.55회
# - 평균 장바구니 수는 두 그룹이 거의 동일
# - 많이 탐색하는 세션일수록 구매보다 비교/고민 단계에 머무를 가능성이 있음
# - 단, 탐색량이 이탈의 원인이라고 단정할 수는 없음

In [39]:
# 세션에서 조회한 상품 수에 따라 장바구니 이후 구매전환율이 어떻게 달라지는지 확인
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            user_session,
            product_id,
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 1
                ELSE 0
            END AS converted
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    ),
    session_stats AS (
        SELECT
            user_session,
            COUNT(DISTINCT product_id) AS session_products
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
          AND event_type = 'view'
        GROUP BY user_session
    )
    SELECT
        CASE
            WHEN s.session_products = 1 THEN '1. 1 product'
            WHEN s.session_products <= 3 THEN '2. 2-3 products'
            WHEN s.session_products <= 5 THEN '3. 4-5 products'
            WHEN s.session_products <= 10 THEN '4. 6-10 products'
            ELSE '5. 11+ products'
        END AS product_range,

        COUNT(*) AS cart_cases,
        SUM(c.converted) AS purchases,
        ROUND(SUM(c.converted) * 100.0 / COUNT(*), 2) AS conversion_rate

    FROM cart_cases c
    JOIN session_stats s
      ON c.user_session = s.user_session

    GROUP BY product_range
    ORDER BY product_range
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬────────────┬───────────┬─────────────────┐
│  product_range   │ cart_cases │ purchases │ conversion_rate │
│     varchar      │   int64    │  int128   │     double      │
├──────────────────┼────────────┼───────────┼─────────────────┤
│ 1. 1 product     │     912278 │    452958 │           49.65 │
│ 2. 2-3 products  │     684748 │    299633 │           43.76 │
│ 3. 4-5 products  │     277870 │    103697 │           37.32 │
│ 4. 6-10 products │     269720 │     89185 │           33.07 │
│ 5. 11+ products  │     195155 │     56140 │           28.77 │
└──────────────────┴────────────┴───────────┴─────────────────┘



In [41]:
# 확인 결과
# - 세션에서 조회한 상품 수가 많아질수록 장바구니 이후 구매전환율이 지속적으로 하락
# - 1개 상품만 본 세션은 49.65%, 11개 이상 본 세션은 28.77%
# - 구매 그룹보다 이탈 그룹의 탐색량이 많았던 이전 결과와 같은 방향의 패턴
# - 많은 상품을 비교하는 탐색형 세션일수록 구매 의도가 상대적으로 낮을 가능성이 있음
# - 단, 조회 상품 수 자체가 이탈의 원인이라고 단정할 수는 없음

In [43]:
# 세션 길이에 따라 장바구니 이후 구매전환율이 어떻게 달라지는지 확인
duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            user_session,
            product_id,
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 1
                ELSE 0
            END AS converted
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    ),
    session_duration AS (
        SELECT
            user_session,
            DATE_DIFF(
                'second',
                MIN(event_time),
                MAX(event_time)
            ) AS session_seconds
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session
    )
    SELECT
        CASE
            WHEN s.session_seconds <= 60 THEN '1. <=1 min'
            WHEN s.session_seconds <= 300 THEN '2. 1-5 min'
            WHEN s.session_seconds <= 600 THEN '3. 5-10 min'
            WHEN s.session_seconds <= 1800 THEN '4. 10-30 min'
            ELSE '5. 30+ min'
        END AS duration_range,

        COUNT(*) AS cart_cases,
        SUM(c.converted) AS purchases,
        ROUND(SUM(c.converted) * 100.0 / COUNT(*), 2) AS conversion_rate

    FROM cart_cases c
    JOIN session_duration s
      ON c.user_session = s.user_session

    GROUP BY duration_range
    ORDER BY duration_range
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬────────────┬───────────┬─────────────────┐
│ duration_range │ cart_cases │ purchases │ conversion_rate │
│    varchar     │   int64    │  int128   │     double      │
├────────────────┼────────────┼───────────┼─────────────────┤
│ 1. <=1 min     │     310624 │     66148 │            21.3 │
│ 2. 1-5 min     │     892531 │    455979 │           51.09 │
│ 3. 5-10 min    │     454308 │    217778 │           47.94 │
│ 4. 10-30 min   │     472726 │    192793 │           40.78 │
│ 5. 30+ min     │     209582 │     68915 │           32.88 │
└────────────────┴────────────┴───────────┴─────────────────┘



In [45]:
# 확인 결과
# - 장바구니 이후 구매전환율은 1~5분 세션에서 51.09%로 가장 높음
# - 5분 이후 세션이 길어질수록 전환율이 점진적으로 하락
# - 30분 이상 세션은 32.88%로 낮은 수준
# - 반면 1분 이하의 매우 짧은 세션도 21.30%로 가장 낮음
# - 따라서 구매전환은 '짧을수록 높다'기보다 적정 탐색 시간이 있는 형태로 보임
# - 세션 길이가 구매의 원인이라고 단정할 수는 없음

In [49]:
'''
1. 가격
   - 고가일수록 이탈한다는 단순 관계 없음

2. 장바구니까지 걸린 시간
   - 빠르게 담을수록 구매전환율이 높음

3. 세션 탐색 상품 수
   - 1개 상품: 49.65%
   - 11개 이상: 28.77%
   → 탐색량이 많을수록 전환율 하락

4. 세션 길이
   - 1~5분에서 최고 51.09%
   - 너무 짧거나 너무 긴 세션은 낮음
'''

'\n1. 가격\n   - 고가일수록 이탈한다는 단순 관계 없음\n\n2. 장바구니까지 걸린 시간\n   - 빠르게 담을수록 구매전환율이 높음\n\n3. 세션 탐색 상품 수\n   - 1개 상품: 49.65%\n   - 11개 이상: 28.77%\n   → 탐색량이 많을수록 전환율 하락\n\n4. 세션 길이\n   - 1~5분에서 최고 51.09%\n   - 너무 짧거나 너무 긴 세션은 낮음\n'

In [53]:
#--------------------------------------------------------------------------------------------------------------------------#

In [55]:
# Tableau용 장바구니 진입 시간 구간별 구매전환율 데이터 생성
# 첫 조회 → 장바구니까지 걸린 시간별 구매전환율
cart_time_conversion = duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            DATE_DIFF('second', first_view_time, first_cart_time) AS seconds_to_cart,
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 1 ELSE 0
            END AS converted
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    )
    SELECT
        CASE
            WHEN seconds_to_cart <= 10 THEN '1. <=10s'
            WHEN seconds_to_cart <= 30 THEN '2. 11-30s'
            WHEN seconds_to_cart <= 60 THEN '3. 31-60s'
            WHEN seconds_to_cart <= 120 THEN '4. 61-120s'
            WHEN seconds_to_cart <= 300 THEN '5. 121-300s'
            ELSE '6. 300s+'
        END AS time_range,
        COUNT(*) AS cart_cases,
        SUM(converted) AS purchases,
        ROUND(SUM(converted) * 100.0 / COUNT(*), 2) AS conversion_rate
    FROM cart_cases
    GROUP BY time_range
    ORDER BY time_range
""").df()

# Tableau용 CSV로 저장
cart_time_conversion.to_csv(
    r"..\data\marts\dashboard_cart_time.csv",
    index=False
)

cart_time_conversion

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,time_range,cart_cases,purchases,conversion_rate
0,1. <=10s,500835,254814.0,50.88
1,2. 11-30s,625144,286219.0,45.78
2,3. 31-60s,405072,160105.0,39.53
3,4. 61-120s,309628,114578.0,37.01
4,5. 121-300s,258921,96172.0,37.14
5,6. 300s+,240171,89725.0,37.36


In [56]:
# Tableau용 세션 조회 상품 수별 장바구니 이후 구매전환율 데이터 생성
# 한 세션에서 조회한 상품 수별 장바구니 이후 전환율
product_count_conversion = duckdb.sql(f"""
    WITH event_times AS (
        SELECT
            user_session,
            product_id,
            MIN(CASE WHEN event_type = 'view' THEN event_time END) AS first_view_time,
            MIN(CASE WHEN event_type = 'cart' THEN event_time END) AS first_cart_time,
            MIN(CASE WHEN event_type = 'purchase' THEN event_time END) AS first_purchase_time
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session, product_id
    ),
    cart_cases AS (
        SELECT
            user_session,
            product_id,
            CASE
                WHEN first_purchase_time IS NOT NULL
                 AND first_cart_time <= first_purchase_time
                THEN 1 ELSE 0
            END AS converted
        FROM event_times
        WHERE first_view_time IS NOT NULL
          AND first_cart_time IS NOT NULL
          AND first_view_time <= first_cart_time
    ),
    session_stats AS (
        SELECT
            user_session,
            COUNT(DISTINCT product_id) AS viewed_products
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
          AND event_type = 'view'
        GROUP BY user_session
    )
    SELECT
        CASE
            WHEN s.viewed_products = 1 THEN '1. 1 product'
            WHEN s.viewed_products <= 3 THEN '2. 2-3 products'
            WHEN s.viewed_products <= 5 THEN '3. 4-5 products'
            WHEN s.viewed_products <= 10 THEN '4. 6-10 products'
            ELSE '5. 11+ products'
        END AS product_range,
        COUNT(*) AS cart_cases,
        SUM(c.converted) AS purchases,
        ROUND(SUM(c.converted) * 100.0 / COUNT(*), 2) AS conversion_rate
    FROM cart_cases c
    JOIN session_stats s
      ON c.user_session = s.user_session
    GROUP BY product_range
    ORDER BY product_range
""").df()

# Tableau용 CSV로 저장
product_count_conversion.to_csv(
    r"..\data\marts\dashboard_product_count.csv",
    index=False
)

product_count_conversion

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,product_range,cart_cases,purchases,conversion_rate
0,1. 1 product,912278,452958.0,49.65
1,2. 2-3 products,684748,299633.0,43.76
2,3. 4-5 products,277870,103697.0,37.32
3,4. 6-10 products,269720,89185.0,33.07
4,5. 11+ products,195155,56140.0,28.77
